# RQ3: controlling weight change — training

How update size and anchoring (E8: peak learning rate × L2-SP strength) and the trainable scope (E9: frozen backbone, DiagScale, LoRA) shape the development of transfer on held-out credit tables, forgetting on the non-credit retention panel, and the distance moved from the pretrained weights.

Monitor points use one fixed context/query split per table (up to 10,000 rows, two preprocessing views); they show how the effect develops and are not the five-fold evaluation.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq3/01_weight_change_training')
report = cp.NotebookReport('RQ3: controlling weight change — training', sink=sink)
frame = c2.load_trajectories(['reference', 'update_size_anchoring', 'scope'])
curves = c2.monitor_curves(c2.monitor_changes(frame))

## 1. Design

The arms of this research question: what each changes versus the reference recipe, the tasks and the number of training runs.

In [2]:
report.table('Arms of RQ2', c2.design_table(2))
report.add('1. Design', 'Each arm changes only the listed settings; E1 is the shared control.')

,experiment,slug,arm,change,tasks,runs,track,models
0,E6,context,—,Does CPT help more when the target table offer...,"PD, LGD",0,NaN,NaN
1,E7,transfer,0,"single_source=0, successful_updates=2500, lear...",NaN,4,pd,4.0
2,E7,transfer,1,"single_source=1, successful_updates=2500, lear...",NaN,4,pd,4.0
3,E7,transfer,2,"single_source=2, successful_updates=2500, lear...",NaN,4,pd,4.0
4,E7,transfer,3,"single_source=3, successful_updates=2500, lear...",NaN,4,pd,4.0
5,E7,transfer,4,"single_source=4, successful_updates=2500, lear...",NaN,4,pd,4.0
6,E7,transfer,5,"single_source=5, successful_updates=2500, lear...",NaN,4,pd,4.0
7,E7,transfer,6,"single_source=6, successful_updates=2500, lear...",NaN,4,pd,4.0
8,E7,transfer,7,"single_source=7, successful_updates=2500, lear...",NaN,4,pd,4.0
9,E7,transfer,8,"single_source=8, successful_updates=2500, lear...",NaN,4,pd,4.0


## 2. Update size and anchoring (E8)

Question: how do larger learning rates and weaker or stronger L2-SP anchoring change transfer, forgetting and the relative weight change during training? Colour orders the learning rate; line style marks the L2-SP strength.

In [3]:
for track in ('pd', 'lgd'):
    sub = c2.select(curves, track=track, slug=['reference', 'update_size_anchoring'])
    drift = c2.drift_curves(c2.select(frame, slug=['reference', 'update_size_anchoring']), track)
    arms = r2.arm_styles(pd.concat([c2.arm_settings('reference', track), c2.arm_settings('update_size_anchoring', track)]), color_by='learning_rate', linestyle_by='l2sp_lambda')
    fig = r2.arm_trajectories([dict(curves=c2.select(sub, scope='test'), x='successful_updates', ylabel='Held-out credit\nΔ ' + r2.SCORE[track]),
                               dict(curves=c2.select(sub, scope='ood'), x='successful_updates', ylabel='Retention panel\nΔ ' + r2.SCORE[track]),
                               dict(curves=drift, x='successful_updates', ylabel='‖w − w₀‖ / ‖w₀‖', log_y=True)],
                              arms, title='Learning rate and L2-SP during training (E8)')
    cp.show(sink, c2.pages(fig, 'update_size_monitor',
        'Development during continued pretraining of every E8 arm and the reference (E1, black: learning rate 3e-7, '
        'λ 0.003), per base model (columns): change from update 0 of the equal-table mean monitor score on the held-out '
        'credit tables (top) and on the non-credit retention panel (middle), and the relative weight change '
        '‖w − w₀‖/‖w₀‖ of all weights (bottom, log scale), each averaged over the four partitions. Colour orders the peak '
        'learning rate; solid, dashed and dotted lines are L2-SP λ = 0.003, 0 and 0.03. Monitor: one fixed context/query '
        'split of up to 10,000 rows per table, not the five-fold evaluation.'), track=track)
    report.table(track.upper() + ' E8 change at the last monitor point', c2.curve_endpoints(sub))
    report.table(track.upper() + ' E8 relative weight change at the last monitor point', c2.curve_endpoints(drift))
report.add('2. Update size and anchoring (E8)', c2.evidence_note('E8 monitor curves', c2.select(curves, slug='update_size_anchoring')))

No measurements available for this section yet.


""


""


No measurements available for this section yet.


""


""


## 3. Trainable scope (E9)

Question: at matched learning-rate pairs, does restricting what changes (frozen backbone, DiagScale query scaling, LoRA on query/value) change transfer or forgetting during training? Weight-change measures differ between scopes and are reported in the cost table only.

In [4]:
for track in ('pd', 'lgd'):
    sub = c2.select(curves, track=track, slug=['reference', 'scope'])
    arms = r2.arm_styles(pd.concat([c2.arm_settings('reference', track), c2.arm_settings('scope', track)]), color_by='trainable', linestyle_by='rate_within_scope')
    fig = r2.arm_trajectories([dict(curves=c2.select(sub, scope='test'), x='successful_updates', ylabel='Held-out credit\nΔ ' + r2.SCORE[track]),
                               dict(curves=c2.select(sub, scope='ood'), x='successful_updates', ylabel='Retention panel\nΔ ' + r2.SCORE[track])],
                              arms, title='Trainable scope during training (E9)')
    cp.show(sink, c2.pages(fig, 'scope_monitor',
        'Development during continued pretraining of the E9 scopes, each at two peak learning rates with L2-SP λ = 0.003 '
        'on the effective weight change, and the full-weight reference (E1, black), per base model (columns): change '
        'from update 0 of the equal-table mean monitor score on the held-out credit tables (top) and on the non-credit '
        'retention panel (bottom), averaged over the four partitions. Colour: trainable scope; dashed and solid: the '
        'lower and the higher learning rate of that scope. Monitor: one fixed context/query split of up to 10,000 rows '
        'per table, not the five-fold evaluation.'), track=track)
    report.table(track.upper() + ' E9 change at the last monitor point', c2.curve_endpoints(sub))
report.add('3. Trainable scope (E9)', c2.evidence_note('E9 monitor curves', c2.select(curves, slug='scope')))

No measurements available for this section yet.


""


No measurements available for this section yet.


""


## 4. Training cost

Wall-clock hours, rows processed, peak reserved GPU memory and the final weight-change measure per arm and model.

In [5]:
report.table('Training cost per arm and model (E1, E8, E9)', c2.resource_summary(c2.run_resources(frame)))
report.add('4. Training cost', c2.evidence_note('training cost records', frame))

""


## Summary

Every displayed table and figure, in section order.

In [6]:
print(report.summary(sink))

RQ3: controlling weight change — training

1. Design
Each arm changes only the listed settings; E1 is the shared control.

Table: Arms of RQ2
CSV (first column: row index)
row,experiment,slug,arm,change,tasks,runs,track,models
0,E6,context,—,"Does CPT help more when the target table offers few labelled context rows, and is what helps credit-specific?","PD, LGD",0,unavailable,unavailable
1,E7,transfer,0,"single_source=0, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
2,E7,transfer,1,"single_source=1, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
3,E7,transfer,2,"single_source=2, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
4,E7,transfer,3,"single_source=3, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
5,E7,transfer,4,"single_source=4, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
6,E7,transfer,5,"single_source=5, successful_updates=2500, learning_rate=1e-06",unavailable,4,pd,4
7,E7,transfer,6,